In [1]:
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from clipBox import *
import datetime

%matplotlib notebook

In [2]:
def null_space(A, rcond = None):
        """
        Computes the null space of a matrix.
        """
        u, s, vh = np.linalg.svd(A, full_matrices=True)
        M, N = u.shape[0], vh.shape[1]
        if rcond is None:
            rcond = np.finfo(s.dtype).eps * max(M, N)
        tol = np.amax(s) * rcond
        num = np.sum(s > tol, dtype=int)
        Q = vh[num:, :].T.conj()
        return Q
    
def norm_points(p):
    r,c = p.shape
    for i in range(c):
        if p[-1,i] != 0:
            p[:,i] /= p[-1,i]
    return p

def time_difference(start_time, end_time):
    """
    Calculates the time difference between two datetime objects in microseconds.
    """
    delta = end_time - start_time
    if delta.seconds == 0:
        return delta.microseconds
    else:
        return delta.seconds*1e6+microseconds




In [3]:
def sortPts(P):
    theta = np.zeros(4)
    mX = np.mean(P[0,: ])
    mY = np.mean(P[1, :])
    for k in range(4):
        Dx = P[0, k] - mX
        Dy = P[1, k] - mY
        theta[k] = np.arctan2(Dy, Dx)
    
    indices = sorted(range(len(theta)), key = lambda index: theta[index])
    sP = P[:, indices]
    return sP, indices

def sortPtsIdx(p,i,j):
    P = np.array([[p[i, 0, 0],p[i, 0, 2],p[j, 0, 0],p[j, 0, 2]],
                  [p[i, 0, 1],p[i, 0, 3],p[j, 0, 1],p[j, 0, 3]],
                  [  1,                1,         1,        1]]).astype('float64')
    return sortPts(P)

def line_similarity(line_a, line_b, threshold=1, normType = 0, clpB=None):
    distance = np.inf
    pts = None
    tstamps = []
    tstamps.append([0,datetime.datetime.now()])
    if normType == 1:
        # Normalize the lines as homogeneous variable
        linea_n = line_a / line_a[2]
        lineb_n = line_b / line_b[2]
        tmp = linea_n[:2] - lineb_n[:2]
        distance = np.dot(tmp, tmp)
        tstamps.append([1, datetime.datetime.now()])
    elif normType == 2:
        # Normalize the lines according to their size
        lineb_n = line_b / np.linalg.norm(line_b)
        linea_n = line_a / np.linalg.norm(line_a)
        tmp = lineb_n - linea_n
        distance = np.dot(tmp, tmp)
        tstamps.append([2, datetime.datetime.now()])
    else:
        if clpB == None:
            clpB = clipBox((0,540),(1920,540))
        pl1, pl2, success = clpB.clipLine(line_a)
        if success == True:
            tstamps.append([2, datetime.datetime.now()])
            pl3, pl4, success = clpB.clipLine(line_b)
            if success == True:
                tstamps.append([3, datetime.datetime.now()])
                P = np.hstack([pl1,pl2,pl3,pl4]).reshape(4,3).transpose()
                tstamps.append([4, datetime.datetime.now()])
                sP, idx = sortPts(P)
                tstamps.append([5, datetime.datetime.now()])
                d=[]
                tmp = sP[:2,0]-sP[:2,1]
                d.append(np.dot(tmp, tmp)) #Squared Distance between P[0,:] and P[1,:]
                tmp = sP[:2,2]-sP[:2,3]
                d.append(np.dot(tmp, tmp)) #Squared Distance between P[0,:] and P[1,:]
                pts = P.copy()
                tstamps.append([6, datetime.datetime.now()])
                if d[0] > d[1]:
                    distance = d[0]
                    pts = np.hstack([pts, np.array(sP[:,0],ndmin=2).transpose()])
                    pts = np.hstack([pts, np.array(sP[:,1],ndmin=2).transpose()])
                else:
                    distance = d[1]
                    pts = np.hstack([pts, np.array(sP[:,2],ndmin=2).transpose()])
                    pts = np.hstack([pts, np.array(sP[:,3],ndmin=2).transpose()])
                tstamps.append([7, datetime.datetime.now()])

    # Compute similarity
    tstamps.append([8, datetime.datetime.now()])
    
    for i in range(1,len(tstamps)):
        tstamps[i][1] = time_difference(tstamps[0][1],tstamps[i][1])
    return distance <= (threshold * threshold), distance, pts, tstamps

In [4]:
def select_lines_with_distance(lines_vp1, clpBox, threshold_min, threshold_max, max_attempts=1000):
    """
    Select two random lines with distance between thresholds.

    Args:
        lines_vp1: List of lines, each as (x1, y1, x2, y2)
        img_width: Image width for line extension
        img_height: Image height for line extension
        threshold_min: Minimum allowed distance between lines
        threshold_max: Maximum allowed distance between lines
        max_attempts: Maximum attempts before giving up

    Returns:
        Tuple of (idx1, idx2, distance) for the selected lines

    Raises:
        ValueError if no suitable pair is found
    """
    n = len(lines_vp1)
    if n < 2:
        raise ValueError("Need at least 2 lines to select a pair")

    threshold_min *= threshold_min
    threshold_max *= threshold_max
        
    print ("Select_Lines_with_distance", flush=True)
    for intento in range(max_attempts):
        print("Intento  #%d" % intento, flush = True)
        # Select two distinct random indices
        idx1, idx2 = random.sample(range(n), 2)
        
        line1 = lines_vp1[idx1][0]
        M = np.array([[line1[0], line1[1], 1], [line1[2], line1[3], 1]])
        l1 = null_space(M)[:, 0]
        l1 /= l1[2]
        
        line2 = lines_vp1[idx2][0]
        M = np.array([[line2[0], line2[1], 1], [line2[2], line2[3], 1]])
        l1 = null_space(M)[:, 0]
        l1 /= l1[2]
        
        # Calculate distance between extended lines
        start_time = datetime.datetime.now()
        _, distance, _, tstamps = line_similarity(l1, l2, threshold=1, normType = 0, clpB=clpBox)
        print("Timestamps: ", tstamps[1:])
        end_time = datetime.datetime.now()
        print("Antes de line_similarity", start_time, flush=True)
        print("Despues de line_similarity:", end_time,flush=True)
        
        print ("La distancia al cuadrado entre l1 y l2 es:", distance, flush=True)
        print ("Y me costó %f microsegundos calcularlo" % time_difference(start_time, end_time), "\n", flush=True)
        
        # Check if distance is within desired range
        if threshold_min <= distance <= threshold_max:
            return idx1, idx2, distance

    raise ValueError(f"No valid line pair found after {max_attempts} attempts")

In [5]:
def line_passes_near_vp(line, vp, threshold=10):
    x1, y1, x2, y2 = line
    line_params = np.polyfit([x1, x2], [y1, y2], 1)
    slope, intercept = line_params
    vp_x, vp_y = vp[0], vp[1]
    return abs(vp_y - (slope * vp_x + intercept)) < threshold

def def_grid_lines(r0,r1,c0,c1,w,h):
    R=np.linspace(r0,r1,r1-r0+1)
    C=np.linspace(c0,c1,c1-c0+1)
    n = len(R)+len(C)
    l=np.zeros((3, n))
    
    #Definimos primero lineas horizontales
    idx=0
    for i in R:
        l[:,idx]=[0, 1, h*i]
        idx +=1
        
    #Definimos primero lineas verticales
    for i in C:
        l[:,idx]=[1, 0, h*i]
        idx=idx+1
    return l

In [6]:
# Definir puntos de fuga y puntos
vp1 = np.array([-3, 540, 1])
vp2 = np.array([1916, 540, 1])

# Load data
lines_near_vps = np.load('merged_lines_near_vps.npy')

clpBox = clipBox((0,0),(1920,1080))

lines_vp1 = []
lines_vp2 = []

img = cv2.imread('vp.jpg')
cv2.circle(img, (vp1[0], vp1[1]), 5, (0, 255, 0), -1)
cv2.circle(img, (vp2[0], vp2[1]), 5, (0, 255, 0), -1)

for line in lines_near_vps:
    if line_passes_near_vp(line[0], vp1):
        lines_vp1.append(line)
    elif line_passes_near_vp(line[0], vp2):
        lines_vp2.append(line)

# Draw lines passing through vp1
for line in lines_vp1:
    x1, y1, x2, y2 = line[0]
    M = np.array([[x1, y1, 1], [x2, y2, 1]])
    l1 = null_space(M)[:, 0]
   
    pt1, pt2, success = clpBox.clipLine(l1)
    if success:
        pt1 /= pt1[2]
        pt2 /= pt2[2]
        
        pt1=tuple(np.round(pt1[:2]).astype('int64'))
        pt2=tuple(np.round(pt2[:2]).astype('int64'))
   
        cv2.line(img, pt1, pt2, (255, 0, 0), 1)  # Blue color for lines passing through vp1

# Draw lines passing through vp2
for line in lines_vp2:
    x1, y1, x2, y2 = line[0]
    M = np.array([[x1, y1, 1], [x2, y2, 1]])
    l2 = np.array(null_space(M)[:, 0])
    pt1, pt2, success = clpBox.clipLine(l2)
    if success:
        pt1 /= pt1[2]
        pt2 /= pt2[2]
    
        pt1=tuple(np.round(pt1[:2]).astype('int64'))
        pt2=tuple(np.round(pt2[:2]).astype('int64'))
   
        cv2.line(img, pt1, pt2[:2], (0, 0, 255), 1)  # Red color for lines passing through vp2
    
# Display the image with the drawn lines
cv2.namedWindow("Image with Lines", cv2.WINDOW_NORMAL)
cv2.imshow('Image with Lines', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

QSettings::value: Empty key passed
QSettings::value: Empty key passed


In [7]:
tuple(pt1[:2])

(1920, 534)

In [15]:
def points_to_homogeneous_line(x1, y1, x2, y2):
    return np.cross([x1, y1, 1], [x2, y2, 1])

def order_lines_by_intersection(lines, horizon_y=540, offset=10):
    H10 = np.array([0, 1, -(horizon_y + offset)])  # Línea horizontal

    intersections = []
    for line in lines:
        intersection = np.cross(line, H10)
        intersection = intersection / intersection[-1]  # Normalizar
        intersections.append(intersection)

    # Ordenar las líneas según la coordenada x de las intersecciones
    sorted_indices = np.argsort([pt[0] for pt in intersections])  # Ordenar por x
    ordered_lines = [lines[i] for i in sorted_indices]
    return ordered_lines

In [20]:
def main_process(lines_near_vps, vp1, vp2, thr=1, iterations=100, grid_size=5, region=[1920, 1080]):
    results = []
    
    lines_vp1 = []
    lines_vp2 = []


    for line in lines_near_vps:
     
        if line_passes_near_vp(line[0], vp1):
            lines_vp1.append(line)
        elif line_passes_near_vp(line[0], vp2):
            lines_vp2.append(line)
   

    #Define Grid Lines
    grid_lines = def_grid_lines(-grid_size, grid_size, -grid_size, grid_size, 1, 1)
    clpBox = clipBox((0,0), (region[0], region[1]))
    
    # Define the square of 1x1
    pr = np.ones((3,4))
    pr[:2,0] = [0., 1]
    pr[:2,1] = [0., 0.]
    pr[:2,2] = [1, 0.]
    pr[:2,3] = [1, 1.]
    pr = norm_points(pr)
    src_pts = np.float32([pr[:, 0], pr[:, 1], pr[:, 2], pr[:, 3]])
    
    for it in range(iterations):
        print ("Iteracion: ", it, flush=True)
        # Step 1: Select random lines
        
        # Select two random lines from lines_vp1
        print ("Step 1.1: Select lines from VP1", flush=True)
        idx1, idx2, distance = select_lines_with_distance(lines_vp1, clpBox, 50, 2000)
        random_lines_vp1 = [lines_vp1[idx1], lines_vp1[idx2]]
        
        # Select two random lines from lines_vp2
        print ("Step 1.2: Select lines from VP2", flush=True)
        idx1, idx2, distance = select_lines_with_distance(lines_vp2, clpBox, 50, 2000)
        random_lines_vp2 = [lines_vp2[idx1], lines_vp2[idx2]]
        
        homogeneous_lines = [points_to_homogeneous_line(*line[0]) for line in random_lines_vp1 + random_lines_vp2]
        # print(homogeneous_lines)
        
        # Step 2: Find intersections
        print("Step 2:ordered_lines_by_intersection", flush=True)
        ordered_lines = order_lines_by_intersection(homogeneous_lines)

        l1 = ordered_lines[1]
        l2 = ordered_lines[0]
        l3 = ordered_lines[2]
        l4 = ordered_lines[3]

        p1 = np.cross(l2, l3)
        p2 = np.cross(l1, l3)
        p3 = np.cross(l1, l4)
        p4 = np.cross(l2, l4)

        p = np.zeros((3, 4))
        p[:, 0] = p1
        p[:, 1] = p2
        p[:, 2] = p3
        p[:, 3] = p4
        p = norm_points(p)

        #Step 3 Compute homography M
        print("Step 3:Compute Homography", flush=True)
        
        dst_pts = np.float32([p[:, 0], p[:, 1], p[:, 2], p[:, 3]])
        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)  

        try:
            Hl = np.linalg.inv(M).T
        except np.linalg.LinAlgError:
            continue

        projected_lines = np.dot(Hl, grid_lines)
        print ("Projected Lines Antes:", projected_lines, flush=True)
        projected_lines = projected_lines / np.linalg.norm(projected_lines, axis=0)
        print ("Projected Lines Despues:", projected_lines, flush=True)

        # Step 4: Compare lines
        similarities = 0
        Dist = 0
        clp = clipBox((0,region[1]//2),(region[0], region[1]//2))
        cont = 0
        print ("Step 4: Compara Lineas", flush=True)
        print ("projected_lines.shape = ", projected_lines.shape,flush=True)
        print ("len(lines_near_vps)   = ", len(lines_near_vps), flush=True)
        for line in projected_lines.T:
            for original_line in lines_near_vps:
                cont+=1
                original_line =original_line[0]
                homogeneous_line = points_to_homogeneous_line(original_line[0], original_line[1], original_line[2], original_line[3])

                isSimil,d,_,_ = line_similarity(line, homogeneous_line, threshold = thr, normType = 0, clpB = clp)
                #isSimil,d,_ = line_similarity_old(line, homogeneous_line, 0.2)
                if isSimil:
                    similarities += 1
                    Dist += d
                    break
        print("Se requirieron ", cont, " iteraciones para buscar similitudes.", flush=True)

        # Step 5: Save results
        results.append((similarities, Dist/similarities, M, it))

    # Step 6: Rank results
    results.sort(key=lambda x: x[0], reverse=True)
    return results

In [21]:
# Run the process
best_results = main_process(lines_near_vps, vp1, vp2, thr=2, iterations=100, grid_size=5, region=(1920,1080))

Iteracion:  0
Step 1.1: Select lines from VP1
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 340], [3, 523], [4, 545], [5, 615], [6, 636], [7, 661], [8, 663]]
Antes de line_similarity 2025-05-18 17:23:37.329968
Despues de line_similarity: 2025-05-18 17:23:37.330681
La distancia al cuadrado entre l1 y l2 es: 3805207.143698069
Y me costó 713.000000 microsegundos calcularlo 

Step 1.2: Select lines from VP2
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 253], [3, 429], [4, 451], [5, 521], [6, 561], [7, 593], [8, 595]]
Antes de line_similarity 2025-05-18 17:23:37.332815
Despues de line_similarity: 2025-05-18 17:23:37.333937
La distancia al cuadrado entre l1 y l2 es: 71432.93033312286
Y me costó 1122.000000 microsegundos calcularlo 

Step 2:ordered_lines_by_intersection
Step 3:Compute Homography
Projected Lines Antes: [[-1.09247154e-03 -1.04625623e-03 -1.00004093e-03 -9.53825624e-04
  -9.07610320e-04 -8.61395015e-04 -8.15179711e-04 -7.68964406e-04
  -7.22749101e-04 -6.76

Projected Lines Despues: [[-1.89993042e-04 -2.01828946e-04 -2.14269543e-04 -2.27362387e-04
  -2.41160154e-04 -2.55721350e-04 -2.71111137e-04 -2.87402312e-04
  -3.04676456e-04 -3.23025302e-04 -3.42552362e-04  2.92725556e-04
   2.94710755e-04  2.98113765e-04  3.05297111e-04  3.30430423e-04
  -1.03759490e-04 -2.54991283e-04 -2.68716241e-04 -2.73862097e-04
  -2.76557327e-04 -2.78215856e-04]
 [ 1.73972986e-03  1.76212088e-03  1.78565584e-03  1.81042472e-03
   1.83652716e-03  1.86407385e-03  1.89318805e-03  1.92400749e-03
   1.95668649e-03  1.99139859e-03  2.02833963e-03  8.14277463e-04
   8.07298366e-04  7.95334866e-04  7.70081348e-04  6.81723548e-04
  -1.47859943e-03 -9.46934720e-04 -8.98683754e-04 -8.80593167e-04
  -8.71117913e-04 -8.65287249e-04]
 [-9.99998469e-01 -9.99998427e-01 -9.99998383e-01 -9.99998335e-01
  -9.99998285e-01 -9.99998230e-01 -9.99998171e-01 -9.99998108e-01
  -9.99998039e-01 -9.99997965e-01 -9.99997884e-01 -9.99999626e-01
  -9.99999631e-01 -9.99999639e-01 -9.99999657e-

Despues de line_similarity: 2025-05-18 17:23:38.113402
La distancia al cuadrado entre l1 y l2 es: 657132.0896697482
Y me costó 1156.000000 microsegundos calcularlo 

Step 2:ordered_lines_by_intersection
Step 3:Compute Homography
Projected Lines Antes: [[-9.86642275e-04 -9.74985161e-04 -9.63328046e-04 -9.51670932e-04
  -9.40013818e-04 -9.28356703e-04 -9.16699589e-04 -9.05042474e-04
  -8.93385360e-04 -8.81728245e-04 -8.70071131e-04  2.20169649e-03
   2.21335360e-03  2.22501072e-03  2.23666783e-03  2.24832494e-03
   2.25998206e-03  2.27163917e-03  2.28329629e-03  2.29495340e-03
   2.30661052e-03  2.31826763e-03]
 [-1.03479785e-01 -7.59140037e-02 -4.83482219e-02 -2.07824402e-02
   6.78334161e-03  3.43491234e-02  6.19149051e-02  8.94806869e-02
   1.17046469e-01  1.44612250e-01  1.72178032e-01 -1.05624178e-01
  -7.80583966e-02 -5.04926149e-02 -2.29268331e-02  4.63894866e-03
   3.22047304e-02  5.97705122e-02  8.73362939e-02  1.14902076e-01
   1.42467857e-01  1.70033639e-01]
 [ 5.62808385e+01 

Projected Lines Despues: [[-5.15595188e-04 -4.30558055e-04 -3.70373398e-04 -3.25535407e-04
  -2.90837912e-04 -2.63190352e-04 -2.40642240e-04 -2.21901853e-04
  -2.06079768e-04 -1.92543557e-04 -1.80831222e-04  1.62768805e-04
   2.60529392e-04  7.70990102e-04  6.58575569e-04  2.14826801e-04
   1.22389105e-04  8.23798442e-05  6.00493802e-05  4.58005250e-05
   3.59183036e-05  2.86621528e-05]
 [ 1.80777644e-03  1.82327429e-03  1.83424281e-03  1.84241443e-03
   1.84873796e-03  1.85377666e-03  1.85788600e-03  1.86130139e-03
   1.86418492e-03  1.86665186e-03  1.86878641e-03 -2.39479564e-03
  -2.73166182e-03 -4.49061804e-03 -4.35424842e-04  1.09366395e-03
   1.41218919e-03  1.55005453e-03  1.62700162e-03  1.67610082e-03
   1.71015331e-03  1.73515680e-03]
 [-9.99998233e-01 -9.99998245e-01 -9.99998249e-01 -9.99998250e-01
  -9.99998249e-01 -9.99998247e-01 -9.99998245e-01 -9.99998243e-01
  -9.99998241e-01 -9.99998239e-01 -9.99998237e-01  9.99997119e-01
   9.99996235e-01  9.99989620e-01 -9.99999688e-

Step 4: Compara Lineas
projected_lines.shape =  (3, 22)
len(lines_near_vps)   =  37
Se requirieron  742  iteraciones para buscar similitudes.
Iteracion:  10
Step 1.1: Select lines from VP1
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 92], [3, 161], [4, 173], [5, 208], [6, 217], [7, 642], [8, 643]]
Antes de line_similarity 2025-05-18 17:23:38.892113
Despues de line_similarity: 2025-05-18 17:23:38.892778
La distancia al cuadrado entre l1 y l2 es: 3695444.9958195467
Y me costó 665.000000 microsegundos calcularlo 

Step 1.2: Select lines from VP2
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 102], [3, 167], [4, 177], [5, 208], [6, 215], [7, 227], [8, 227]]
Antes de line_similarity 2025-05-18 17:23:38.894883
Despues de line_similarity: 2025-05-18 17:23:38.895130
La distancia al cuadrado entre l1 y l2 es: 909772.4939062493
Y me costó 247.000000 microsegundos calcularlo 

Step 2:ordered_lines_by_intersection
Step 3:Compute Homography
Projected Lines Antes: [[ 9.46820207

...

Step 4: Compara Lineas
projected_lines.shape =  (3, 22)
len(lines_near_vps)   =  37
Se requirieron  761  iteraciones para buscar similitudes.
Iteracion:  89
Step 1.1: Select lines from VP1
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 152], [3, 250], [4, 269], [5, 316], [6, 335], [7, 357], [8, 358]]
Antes de line_similarity 2025-05-18 17:23:52.617057
Despues de line_similarity: 2025-05-18 17:23:52.617455
La distancia al cuadrado entre l1 y l2 es: 3702509.1311227544
Y me costó 398.000000 microsegundos calcularlo 

Step 1.2: Select lines from VP2
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 148], [3, 251], [4, 273], [5, 329], [6, 373], [7, 450], [8, 455]]
Antes de line_similarity 2025-05-18 17:23:52.619090
Despues de line_similarity: 2025-05-18 17:23:52.619623
La distancia al cuadrado entre l1 y l2 es: 158.42682546954876
Y me costó 533.000000 microsegundos calcularlo 

Intento  #1
Timestamps:  [[2, 277], [3, 461], [4, 492], [5, 584], [6, 664], [7, 701], [8, 703]]
A

Step 4: Compara Lineas
projected_lines.shape =  (3, 22)
len(lines_near_vps)   =  37
Se requirieron  741  iteraciones para buscar similitudes.
Iteracion:  91
Step 1.1: Select lines from VP1
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 362], [3, 566], [4, 590], [5, 667], [6, 952], [7, 1001], [8, 1003]]
Antes de line_similarity 2025-05-18 17:23:52.971344
Despues de line_similarity: 2025-05-18 17:23:52.972404
La distancia al cuadrado entre l1 y l2 es: 3690957.730795783
Y me costó 1060.000000 microsegundos calcularlo 

Step 1.2: Select lines from VP2
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 209], [3, 377], [4, 394], [5, 465], [6, 521], [7, 554], [8, 555]]
Antes de line_similarity 2025-05-18 17:23:52.974883
Despues de line_similarity: 2025-05-18 17:23:52.975501
La distancia al cuadrado entre l1 y l2 es: 906947.3087351547
Y me costó 618.000000 microsegundos calcularlo 

Step 2:ordered_lines_by_intersection
Step 3:Compute Homography
Projected Lines Antes: [[-2.03609

Projected Lines Despues: [[-1.26580233e-05 -1.79935184e-05 -2.92430330e-05 -6.84640668e-05
  -2.93255561e-04 -4.99133132e-05 -2.82300663e-05 -2.01173169e-05
  -1.58717914e-05 -1.32605290e-05 -1.14922014e-05  4.02646511e-04
   1.33540690e-03  1.00007499e-03  3.61945976e-04  2.20258745e-04
   1.57930154e-04  1.22874029e-04  1.00403560e-04  8.47725762e-05
   7.32708711e-05  6.44530922e-05]
 [-1.84541539e-03 -1.84530014e-03 -1.84505713e-03 -1.84420988e-03
   1.85202358e-03  1.84676704e-03  1.84629865e-03  1.84612340e-03
   1.84603169e-03  1.84597528e-03  1.84593708e-03 -3.22884750e-03
  -6.45135115e-03 -1.61731826e-03  5.87315456e-04  1.07682087e-03
   1.29215546e-03  1.41326831e-03  1.49089989e-03  1.54490222e-03
   1.58463861e-03  1.61510250e-03]
 [ 9.99998297e-01  9.99998297e-01  9.99998297e-01  9.99998297e-01
  -9.99998242e-01 -9.99998293e-01 -9.99998295e-01 -9.99998296e-01
  -9.99998296e-01 -9.99998296e-01 -9.99998296e-01  9.99994706e-01
   9.99978298e-01 -9.99998192e-01 -9.99999762e-

Despues de line_similarity: 2025-05-18 17:23:53.986498
La distancia al cuadrado entre l1 y l2 es: 3695253.712582726
Y me costó 706.000000 microsegundos calcularlo 

Step 1.2: Select lines from VP2
Select_Lines_with_distance
Intento  #0
Timestamps:  [[2, 302], [3, 388], [4, 406], [5, 454], [6, 468], [7, 485], [8, 485]]
Antes de line_similarity 2025-05-18 17:23:53.989238
Despues de line_similarity: 2025-05-18 17:23:53.990019
La distancia al cuadrado entre l1 y l2 es: 1.2924697071141057e-26
Y me costó 781.000000 microsegundos calcularlo 

Intento  #1
Timestamps:  [[2, 103], [3, 177], [4, 190], [5, 231], [6, 240], [7, 592], [8, 593]]
Antes de line_similarity 2025-05-18 17:23:53.991551
Despues de line_similarity: 2025-05-18 17:23:53.992170
La distancia al cuadrado entre l1 y l2 es: 160.50881404290877
Y me costó 619.000000 microsegundos calcularlo 

Intento  #2
Timestamps:  [[2, 93], [3, 166], [4, 175], [5, 205], [6, 213], [7, 226], [8, 227]]
Antes de line_similarity 2025-05-18 17:23:53.9934

Projected Lines Despues: [[ 1.07511540e-04  1.26760148e-04  1.52343582e-04  1.88005037e-04
   2.41160135e-04  3.28864333e-04  5.01018208e-04  9.92840899e-04
   1.42451099e-02  1.22088896e-03  6.00908429e-04  1.41521730e-04
   2.47633354e-04  7.04936887e-04  1.14906913e-03  3.46356397e-04
   2.13767726e-04  1.59270902e-04  1.29569535e-04  1.10877421e-04
   9.80305104e-05  8.86577522e-05]
 [-1.77941672e-03 -1.78764200e-03 -1.79857426e-03 -1.81381304e-03
  -1.83652716e-03 -1.87400472e-03 -1.94756907e-03 -2.15773314e-03
  -7.82046299e-03  1.21176538e-03  1.47669555e-03 -2.30952664e-03
  -2.67624308e-03 -4.25665808e-03 -2.15071176e-03  6.23440932e-04
   1.08166220e-03  1.27000098e-03  1.37264764e-03  1.43724679e-03
   1.48164516e-03  1.51403700e-03]
 [ 9.99998411e-01  9.99998394e-01  9.99998371e-01  9.99998337e-01
   9.99998285e-01  9.99998190e-01  9.99997978e-01  9.99997179e-01
   9.99867950e-01 -9.99998521e-01 -9.99998729e-01  9.99997323e-01
   9.99996388e-01  9.99990692e-01 -9.99997027e-

In [22]:
# Print the top 5 results
for i, (similarities, d, M, it) in enumerate(best_results[:5]):
    print(f"Rank {i+1}: Similarities = {similarities}")
    print("Dist promedio: ", d)
    print("Iteracion: ", it)
    print("Matrix M:")
    print(M)

Rank 1: Similarities = 5
Dist promedio:  0.5579095011740847
Iteracion:  14
Matrix M:
[[ 1.41643214e+03 -1.00333053e+01  6.80901794e+02]
 [ 6.57958347e+02 -2.90491365e+00  6.28533203e+02]
 [ 8.64272559e-01 -5.33630074e-03  1.00000000e+00]]
Rank 2: Similarities = 5
Dist promedio:  0.40679242298268353
Iteracion:  39
Matrix M:
[[1.21178053e+02 3.82584352e+00 1.49839868e+03]
 [8.91512299e+01 1.11834594e+00 7.77210510e+02]
 [1.29574260e-01 2.02390419e-03 1.00000000e+00]]
Rank 3: Similarities = 5
Dist promedio:  0.9229356296834104
Iteracion:  42
Matrix M:
[[ 4.69248793e+03 -2.48322374e+00  9.12593811e+02]
 [ 2.78498477e+03 -6.97063186e-01  6.69004028e+02]
 [ 3.92772894e+00 -1.28276085e-03  1.00000000e+00]]
Rank 4: Similarities = 5
Dist promedio:  0.4591860003041178
Iteracion:  46
Matrix M:
[[ 1.99538823e+00 -1.25596805e+03  1.57494739e+03]
 [-3.72712186e+01 -3.52850372e+02  5.84052612e+02]
 [-6.89309284e-02 -6.52315332e-01  1.00000000e+00]]
Rank 5: Similarities = 5
Dist promedio:  0.407579405

In [23]:
for i, (similarities, d, M, it) in enumerate(best_results[:5]):
    img = cv2.imread('vp.jpg')
    lines = def_grid_lines(-5,5,-5,5,1,1)
    Hl = np.linalg.inv(M).T
    projected_lines = np.dot(Hl, lines)
    projected_lines = projected_lines / np.linalg.norm(projected_lines, axis=0)
    # Dibujar líneas proyectadas
    for i in range(projected_lines.shape[1]):
        a, b, c = projected_lines[:, i]
        # Encontrar dos puntos en la línea (para dibujar)
        x0, y0 = 0, int(-c/b) if b != 0 else 0
        x1, y1 = img.shape[1], int((-c - a*img.shape[1])/b) if b != 0 else 0
        cv2.line(img, (x0, y0), (x1, y1), (255, 0, 0), 2)

    cv2.namedWindow("Projected Grid", cv2.WINDOW_NORMAL)
    cv2.imshow("Projected Grid", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()